# 07 — Graph Representation Comparison

## Objective
Compare pure tabular behavioural features against graph-augmented features (degree + shared-entity).  
We measure the lift in validation PR-AUC / ROC-AUC when structural signals are added.


In [ ]:

from pathlib import Path
import sys
import numpy as np
import pandas as pd
from data_utils import load_processed
from anomaly import fit_isolation_forest
from evaluation import evaluate_anomaly_scores, summary_table

ROOT = Path.cwd()
if not (ROOT / "data").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

X_train = load_processed("X_train")
X_val   = load_processed("X_val")
y_val   = X_val["class"].values

# Feature groups
graph_cols = [c for c in X_train.columns if any(k in c for k in ["degree", "users_on", "ips_on", "tx_count"])]
tab_cols   = [c for c in X_train.columns if c not in graph_cols and c != "class"]

print("Tabular features:", len(tab_cols))
print("Graph features:", len(graph_cols))

results = {}
for label, cols in [("Tabular only", tab_cols), ("Graph only", graph_cols), ("Tabular + Graph", tab_cols+graph_cols)]:
    model = fit_isolation_forest(X_train[cols], contamination=0.09, n_estimators=150)
    s = model.predict_anomaly_score(X_val[cols])
    results[label] = evaluate_anomaly_scores(y_val, s)
print(summary_table(results))
